Subject: ST 554 - Homework 9

Name: Franklin Zhou

Date: 4/4/2026

# Homework 9

For this homework you will create a gitHub repo (or use the one from previous mework) and save a python notebook (`.ipynb` file) there. You must use our JupyterHub for this assignment as we’ll be using `pyspark`. You’ll then submit a link to your gitHub repo.

You are tasked with:

- Finding a data set you can fit supervised learning models with (you cannot use the datasets we’ve been using in class - there are many places with free data out there such as the UCI machine learning repository and kaggle).
- **Please read your data in via a URL or include the data as a file in your submission.**
- Using a numeric or binary response, fitting three different classes of models and choosing an overall best model.
- Writing a narrative (via a notebook) with explanations and discussions as you go through the above.

### 0. Read data

The data set that I'm going to use in this homework is the *Seoul Bike Sharing Demand* https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand

The dataset contains weather information (Temperature, Humidity, Windspeed, Visibility, Dewpoint, Solar radiation, Snowfall, Rainfall), the number of bikes rented per hour and date information. 

In [5]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

bike_data = pd.read_csv("SeoulBikeData.csv")

df = spark.createDataFrame(bike_data)
df.show(5)

+----------+-----------------+----+-----------+--------+----------+----------+---------------------+---------------+--------+--------+-------+----------+---------------+
|      Date|Rented Bike Count|Hour|Temperature|Humidity|Wind speed|Visibility|Dew point temperature|Solar Radiation|Rainfall|Snowfall|Seasons|   Holiday|Functioning Day|
+----------+-----------------+----+-----------+--------+----------+----------+---------------------+---------------+--------+--------+-------+----------+---------------+
|01/12/2017|              254|   0|       -5.2|      37|       2.2|      2000|                -17.6|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes|
|01/12/2017|              204|   1|       -5.5|      38|       0.8|      2000|                -17.6|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes|
|01/12/2017|              173|   2|       -6.0|      39|       1.0|      2000|                -17.7|            0.0|     0.0|     0.0| Winter|No Holid

## 1. Splitting the Data, Metrics, and Models

- Using spark MLlib, split the data into a training and test set.
-  Choose and describe a metric you’ll be using to judge your models.
-  You’ll be fitting three different classes of models (of your choice - they can be ones we used in class or ones we didn’t cover - see https://spark.apache.org/docs/latest/ml-classification-regression.html for a list of models they have in MLlib). Briefly describe each model (no code or anything here, just concepts and ideas about what the models are doing). These discussions should be clear to someone that knows statistics but doesn’t know the modeling type/algorithm!

### 1.1. Data splitting

I choose `Rented Bike Count` as response variable and remove the `Date` column since it does not provide useful information to the model but serves just as an index.

In [6]:
train, test = df.drop("Date").randomSplit([0.8, 0.2], seed = 1)

### 1.2. Metric selection

Since the response variable is a continuous type, RMSE (Root Mean Squared Error) is an appropriate metric to compare the models.

$$
RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2}
$$

where $y_i$ is the observed value and $\hat{y}_i$ is the predicted value

### 1.3. Three Models

Under the "Regression" Category, I choose the following three models to fit:

1. Linear regression: https://spark.apache.org/docs/latest/ml-classification-regression.html#linear-regression

The linear regression is to use build a linear model with a response variable and several predictor variables

$$
Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p + \varepsilon
$$

The estimation of parameters $\beta = (\beta_0, \beta_1, \cdots, \beta_p)$ are achieved via ordinary least squares:

$$
\hat{\beta} = \arg\min_{\beta} \sum_{i=1}^{n} (y_i - x_i^\top \beta)^2.
$$


2. Random forest regression: https://spark.apache.org/docs/latest/ml-classification-regression.html#random-forest-regression

Random forest regression is an ensemble model that constructs a large collection of decision trees and aggregates their predictions to produce a more stable and accurate estimator of a continuous response variable.

A random forest consists of $B$ regression trees, each trained on a bootstrap sample of the data. At each split, the random forest algorithm selects a random subset of predictors. Then it averages over $B$ trees to get the final model. 

3. Gradient-boosted tree regression https://spark.apache.org/docs/latest/ml-classification-regression.html#gradient-boosted-tree-regression

Instead of training one powerful model, GBT trains trees one at a time, where each new tree corrects the errors of the previous ensemble.

- starts with a simple prediction, 
- then computes residuals, 
- fits a new tree to those residuals (not the original targets), 
- adds the tree to the ensemble with a small weight (learning rate), 
- and repeats until we have $N$ trees or error stops improving.



## 2. Model Fitting

Next, you should use Spark MLlib to fit your three different classes models to the training data. **This should be done using pipelines** and cross validation to choose your best model for each model type. You should compare your models using your metric chosen earlier.

Notes:

- You should set up a pipeline in pyspark for each of your models
- You should do your transformations using the functions from MLlib to easily put them into the pipeline. **At least one of the pipelines should use four or more transformations** prior to the model fit (`estimator`)
    - VectorAssembler counts as a transformation
    - Doing something like a log transform counts as well
    - Adding polynomial terms or interaction terms counts
    - etc.
- You can use the same set of transformations for multiple models (if appropriate)

### 2.1. Build pipelines

The following transformations are included:

- StringIndexer()
- SQLTransformer()
- VectorAssembler()
- PolynomialExpansion()
- StandardScaler()

In [7]:
from pyspark.ml.feature import StringIndexer, SQLTransformer, VectorAssembler, PolynomialExpansion, StandardScaler

In [9]:
# Step 1: Use StringIndexer() to take the string to a numeric index
indexer = StringIndexer(inputCols = ["Seasons", "Holiday", "Functioning Day"], outputCols = ["Seasons_ind", "Holiday_ind", "Functioning_Day_ind"])
indexerTrans = indexer.fit(df) # to get a .transform() method

indexerTrans.transform(df).show(5)

+----------+-----------------+----+-----------+--------+----------+----------+---------------------+---------------+--------+--------+-------+----------+---------------+-----------+-----------+-------------------+
|      Date|Rented Bike Count|Hour|Temperature|Humidity|Wind speed|Visibility|Dew point temperature|Solar Radiation|Rainfall|Snowfall|Seasons|   Holiday|Functioning Day|Seasons_ind|Holiday_ind|Functioning_Day_ind|
+----------+-----------------+----+-----------+--------+----------+----------+---------------------+---------------+--------+--------+-------+----------+---------------+-----------+-----------+-------------------+
|01/12/2017|              254|   0|       -5.2|      37|       2.2|      2000|                -17.6|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes|        3.0|        0.0|                0.0|
|01/12/2017|              204|   1|       -5.5|      38|       0.8|      2000|                -17.6|            0.0|     0.0|     0.0| Winter|No

In [10]:
# Step 2: 
# select required variables; 
# create log value of Visibility and log value of Rented Bike Count; 
# create precipitation variable: if there is any rain or snow, set the value to 1 otherwise to 0

sqlTrans = SQLTransformer(
    statement = """
                SELECT Hour,
                    Temperature, 
                    Humidity, 
                    `Wind speed`, 
                    log(Visibility + 1) as log_Visibility,                
                    CASE WHEN Rainfall > 0 OR Snowfall > 0 THEN 1 ELSE 0 END AS precipitation,
                    Seasons_ind,
                    Holiday_ind,
                    Functioning_Day_ind,
                    log(`Rented Bike Count` + 1) as label -- This is response variable, marked as label
                FROM __THIS__
                """)

sqlTrans.transform(
    indexerTrans.transform(df)
).show(5)

+----+-----------+--------+----------+-----------------+-------------+-----------+-----------+-------------------+------------------+
|Hour|Temperature|Humidity|Wind speed|   log_Visibility|precipitation|Seasons_ind|Holiday_ind|Functioning_Day_ind|             label|
+----+-----------+--------+----------+-----------------+-------------+-----------+-----------+-------------------+------------------+
|   0|       -5.2|      37|       2.2|7.601402334583733|            0|        3.0|        0.0|                0.0| 5.541263545158426|
|   1|       -5.5|      38|       0.8|7.601402334583733|            0|        3.0|        0.0|                0.0|5.3230099791384085|
|   2|       -6.0|      39|       1.0|7.601402334583733|            0|        3.0|        0.0|                0.0| 5.159055299214529|
|   3|       -6.2|      40|       0.9|7.601402334583733|            0|        3.0|        0.0|                0.0|  4.68213122712422|
|   4|       -6.0|      36|       2.3|7.601402334583733|      

In [12]:
# Step 3: assemble 
assembler = VectorAssembler(inputCols = ["Hour", "Temperature", "Humidity", "Wind speed", "log_Visibility", "precipitation", "Seasons_ind", "Holiday_ind", "Functioning_Day_ind"], 
                            outputCol = "raw_features", 
                            handleInvalid = 'keep')

assembler.transform(
    sqlTrans.transform(
        indexerTrans.transform(df)
    )
).show(5)

+----+-----------+--------+----------+-----------------+-------------+-----------+-----------+-------------------+------------------+--------------------+
|Hour|Temperature|Humidity|Wind speed|   log_Visibility|precipitation|Seasons_ind|Holiday_ind|Functioning_Day_ind|             label|        raw_features|
+----+-----------+--------+----------+-----------------+-------------+-----------+-----------+-------------------+------------------+--------------------+
|   0|       -5.2|      37|       2.2|7.601402334583733|            0|        3.0|        0.0|                0.0| 5.541263545158426|[0.0,-5.2,37.0,2....|
|   1|       -5.5|      38|       0.8|7.601402334583733|            0|        3.0|        0.0|                0.0|5.3230099791384085|[1.0,-5.5,38.0,0....|
|   2|       -6.0|      39|       1.0|7.601402334583733|            0|        3.0|        0.0|                0.0| 5.159055299214529|[2.0,-6.0,39.0,1....|
|   3|       -6.2|      40|       0.9|7.601402334583733|            0|

In [13]:
# Step 4: add Polynomial items
poly = PolynomialExpansion(degree = 2, inputCol = "raw_features", outputCol = "poly_features")

poly.transform(
    assembler.transform(
        sqlTrans.transform(
            indexerTrans.transform(df)
        )
    )
).show(5)

+----+-----------+--------+----------+-----------------+-------------+-----------+-----------+-------------------+------------------+--------------------+--------------------+
|Hour|Temperature|Humidity|Wind speed|   log_Visibility|precipitation|Seasons_ind|Holiday_ind|Functioning_Day_ind|             label|        raw_features|       poly_features|
+----+-----------+--------+----------+-----------------+-------------+-----------+-----------+-------------------+------------------+--------------------+--------------------+
|   0|       -5.2|      37|       2.2|7.601402334583733|            0|        3.0|        0.0|                0.0| 5.541263545158426|[0.0,-5.2,37.0,2....|[0.0,0.0,-5.2,0.0...|
|   1|       -5.5|      38|       0.8|7.601402334583733|            0|        3.0|        0.0|                0.0|5.3230099791384085|[1.0,-5.5,38.0,0....|[1.0,1.0,-5.5,-5....|
|   2|       -6.0|      39|       1.0|7.601402334583733|            0|        3.0|        0.0|                0.0| 5.159

In [14]:
# Step 5: Scale data
scaler = StandardScaler(inputCol = "poly_features", outputCol = "features")

scalerTrans = scaler.fit(
    poly.transform(
        assembler.transform(
            sqlTrans.transform(
                indexerTrans.transform(df)
            )
        )
    )
) # to get a .transform() method

scalerTrans.transform(
    poly.transform(
        assembler.transform(
            sqlTrans.transform(
                indexerTrans.transform(df)
            )
        )
    )
).show(5)

+----+-----------+--------+----------+-----------------+-------------+-----------+-----------+-------------------+------------------+--------------------+--------------------+--------------------+
|Hour|Temperature|Humidity|Wind speed|   log_Visibility|precipitation|Seasons_ind|Holiday_ind|Functioning_Day_ind|             label|        raw_features|       poly_features|            features|
+----+-----------+--------+----------+-----------------+-------------+-----------+-----------+-------------------+------------------+--------------------+--------------------+--------------------+
|   0|       -5.2|      37|       2.2|7.601402334583733|            0|        3.0|        0.0|                0.0| 5.541263545158426|[0.0,-5.2,37.0,2....|[0.0,0.0,-5.2,0.0...|[0.0,0.0,-0.43533...|
|   1|       -5.5|      38|       0.8|7.601402334583733|            0|        3.0|        0.0|                0.0|5.3230099791384085|[1.0,-5.5,38.0,0....|[1.0,1.0,-5.5,-5....|[0.14445477786121...|
|   2|       -6

### 2.2 Build Models

#### 2.2.1. Linear regression Model

In [15]:
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

In [16]:
# Setup linear regression
lr = LinearRegression()

# Setup parameter grid
paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, np.linspace(0.01, 0.2, 10)) \
    .addGrid(lr.elasticNetParam, np.linspace(0.01, 0.2, 10)) \
    .build()

# Setup pipeline
pipeline_lr = Pipeline(stages = [indexerTrans, sqlTrans, assembler, poly, scalerTrans, lr])

# Create cross validation instance
crossval_lr = CrossValidator(estimator = pipeline_lr,
                          estimatorParamMaps = paramGrid_lr,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [17]:
lr_cvModel = crossval_lr.fit(train)

26/04/05 21:52:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


#### 2.2.2. Random forest regression Model

In [18]:
# Setup random forest
rf = RandomForestRegressor()

# Setup parameter grid
paramGrid_rf = ParamGridBuilder() \
    .addGrid(rf.maxDepth, [3, 4, 5, 6]) \
    .build()

# Setup pipeline
pipeline_rf = Pipeline(stages = [indexerTrans, sqlTrans, assembler, poly, scalerTrans, rf])

# Create cross validation instance
crossval_rf = CrossValidator(estimator = pipeline_rf,
                          estimatorParamMaps = paramGrid_rf,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds = 5)

In [19]:
rf_cvModel = crossval_rf.fit(train)

#### 2.2.3. Gradient-boosted tree regression Model

In [20]:
# Setup gradient-boosted tree
gbt = GBTRegressor()

# Setup parameter grid
paramGrid_gbt = ParamGridBuilder() \
    .addGrid(rf.maxDepth, [3, 4, 5, 10, 15]) \
    .build()

# Setup pipeline
pipeline_gbt = Pipeline(stages = [indexerTrans, sqlTrans, assembler, poly, scalerTrans, gbt])

# Create cross validation instance
crossval_gbt = CrossValidator(estimator = pipeline_gbt,
                          estimatorParamMaps = paramGrid_gbt,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [21]:
gbt_cvModel = crossval_gbt.fit(train)

## 3. Model Testing

Lastly, you should evaluate the best models from each class on the test set and state which overall model was deemed the best.

In [22]:
# rmse of linear regression model on test dataset
LR_test_rmse = RegressionEvaluator(metricName='rmse').evaluate(lr_cvModel.transform(test))

# rmse of random forest model on test dataset
RF_test_rmse = RegressionEvaluator(metricName='rmse').evaluate(rf_cvModel.transform(test))

# rmse of gradient-boosted tree regression model on test dataset
GBT_test_rmse = RegressionEvaluator(metricName='rmse').evaluate(gbt_cvModel.transform(test))

print(LR_test_rmse, RF_test_rmse, GBT_test_rmse)


0.6206450273603519 0.5161362078865653 0.47158901273226705


From the output, the gradient-boosted tree regression model has the lowest RMSE value on the test dataset. So we choose this model as the best one.

## 4. Submit

- Make sure all cells are run in your notebook
- Submit the link to your github repo

## 5. References

https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.PolynomialExpansion.html

https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StandardScaler.html

https://spark.apache.org/docs/latest/ml-classification-regression.html#random-forest-regression

https://spark.apache.org/docs/latest/ml-classification-regression.html#gradient-boosted-tree-regression

